# Atlassian Notebook: ML Coding Revision

Rehearse the smallest set of code patterns and interview behaviors supported by the recruiting guidance: weighted sampling, coupon recommendation, tests, scale, and clear tradeoffs.

- **Study time:** 45-60 minutes (or use the 20-minute emergency pass at the end)
- **Prerequisites:** basic Python, array shapes, and probability
- **Mode:** `quick`
- **Data policy:** no files, network calls, or downloads; all examples are tiny and deterministic
- **Provenance:** distilled from `~/Downloads/Atlassian Durga ML Coding Guidance__whisperx_whisper-large-v3-turbo__diarized.txt`; see the [source decision](../../sources/atlassian-ml-coding-guidance.md)

**Scope note:** the recruiter confirmed an ML-engineering coding round rather than general DSA and named coupon recommendation and weighted sampling as examples. NumPy/pandas familiarity was asked about but **not confirmed**, so the core uses standard Python + NumPy and the pandas section is optional insurance. AI/code-completion tools are not allowed in the round.

Run top-to-bottom once. On the second pass, hide the solutions and implement each prompt from memory. Every retained output is labeled. Comments call out intent, invariants, shapes, subtle behavior, and configuration side effects rather than narrating obvious syntax.

## 0. What the round is measuring

Before coding, turn ambiguity into an explicit contract:

1. **Clarify:** objective, input/output shapes, constraints, replacement/duplicates, tie behavior, invalid inputs, and expected scale.
2. **State a baseline:** give the simplest correct approach and its time/space cost.
3. **Implement in small pieces:** validation, core logic, and presentation should be separable.
4. **Test while moving:** normal case, boundary case, invalid input, and one scale-sensitive case.
5. **Adapt aloud:** explain what changes at 10x or 1,000x scale and the broader product/system consequence.

A useful opening sentence is: *“I’ll confirm the contract, write a correct baseline, test its edge cases, then optimize the bottleneck if the constraints require it.”*

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd

np.set_printoptions(
    precision=3, suppress=True
)  # Display only: 3-digit precision and no scientific notation; underlying values are unchanged.


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


def assert_raises(error_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__} from {function.__name__}")


show("Environment | library versions", {"numpy": np.__version__, "pandas": pd.__version__})


--- Environment | library versions ---
{'numpy': '2.5.2', 'pandas': '3.0.5'}


## 1. Weighted sampling

**Prompt:** Given non-negative weights, return sampled item indices.

Clarify before typing:

- With or without replacement? Is order meaningful?
- What should zero, negative, `NaN`, or all-zero weights do?
- Do we need reproducibility? Are weights reused across many calls?
- Return positions, item IDs, or items? How large are item and draw counts?

The implementation below makes those choices explicit. With replacement it uses a CDF plus binary search. Without replacement it uses Gumbel top-k, which produces a weighted ordering over positive-weight items.

In [2]:
def weighted_sample_indices(
    weights,
    n_samples,
    *,
    replace=True,
    rng=None,
):
    """Sample positions in a 1D non-negative weight vector.

    Args:
        weights: Finite, non-negative weights. They need not sum to one.
        n_samples: Number of positions to return.
        replace: Whether the same positive-weight position may repeat.
        rng: Optional ``np.random.Generator`` for reproducibility.
    """
    weights = np.asarray(weights, dtype=np.float64)
    if weights.ndim != 1:
        raise ValueError("weights must be one-dimensional")
    if isinstance(n_samples, (bool, np.bool_)) or not isinstance(n_samples, (int, np.integer)):
        raise TypeError("n_samples must be an integer")
    if n_samples < 0:
        raise ValueError("n_samples must be non-negative")
    if not np.all(np.isfinite(weights)) or np.any(weights < 0):
        raise ValueError("weights must be finite and non-negative")
    if n_samples == 0:
        return np.empty(0, dtype=np.int64)

    positive = weights > 0
    if not np.any(positive):
        raise ValueError("at least one weight must be positive")

    generator = (
        np.random.default_rng() if rng is None else rng
    )  # Honor a caller-supplied generator instead of silently reseeding.
    if replace:
        # Scale before summing so very large finite weights do not overflow.
        scaled = weights / weights.max()
        probabilities = scaled / scaled.sum()
        cdf = np.cumsum(probabilities)
        cdf[-1] = 1.0  # Guard against the final floating-point roundoff.
        uniforms = generator.random(n_samples)
        return np.searchsorted(
            cdf, uniforms, side="right"
        )  # Map each draw to the first cumulative boundary above it.

    positive_indices = np.flatnonzero(positive)
    if n_samples > positive_indices.size:
        raise ValueError("cannot sample more positive-weight items without replacement")

    keys = np.log(weights[positive_indices]) + generator.gumbel(
        size=positive_indices.size
    )  # One perturbed log-weight per positive item; larger keys rank first.
    if n_samples == positive_indices.size:
        local_order = np.argsort(-keys, kind="stable")
    else:
        candidates = np.argpartition(-keys, kth=n_samples - 1)[
            :n_samples
        ]  # Select winners without fully sorting every item.
        local_order = candidates[
            np.argsort(-keys[candidates], kind="stable")
        ]  # Sort only the selected candidates.
    return positive_indices[
        local_order
    ]  # Map local positive-only positions back to the original vector.

In [3]:
weights = np.array([0.0, 1.0, 3.0])
draws = weighted_sample_indices(weights, 40_000, replace=True, rng=np.random.default_rng(7))
empirical = (
    np.bincount(draws, minlength=weights.size) / draws.size
)  # Convert draw counts into empirical probabilities.
without_replacement = weighted_sample_indices(
    [0.0, 1.0, 3.0, 6.0], 3, replace=False, rng=np.random.default_rng(7)
)

assert np.allclose(
    empirical, [0.0, 0.25, 0.75], atol=0.012
)  # Sampling tests need tolerance, not exact equality.
assert len(np.unique(without_replacement)) == 3
assert 0 not in without_replacement
assert weighted_sample_indices([1.0], 0).shape == (0,)
assert_raises(ValueError, weighted_sample_indices, [-1.0, 2.0], 1)
assert_raises(ValueError, weighted_sample_indices, [0.0, 0.0], 1)
assert_raises(ValueError, weighted_sample_indices, [1.0, 0.0], 2, replace=False)

show(
    "Weighted sampling | target vs empirical probabilities", np.vstack(([0, 0.25, 0.75], empirical))
)
show("Weighted sampling | seeded draw without replacement", without_replacement)
show(
    "Weighted sampling | test status",
    "distribution, uniqueness, zero, and invalid-input checks passed",
)


--- Weighted sampling | target vs empirical probabilities ---
[[0.    0.25  0.75 ]
 [0.    0.247 0.753]]

--- Weighted sampling | seeded draw without replacement ---
[3 2 1]

--- Weighted sampling | test status ---
distribution, uniqueness, zero, and invalid-input checks passed


### Weighted-sampling tradeoffs

- **CDF method:** `O(m)` setup for `m` items, then `O(s log m)` for `s` draws; cache the CDF when the same weights are reused.
- **Alias table:** `O(m)` setup and `O(1)` per replacement draw; useful only when many draws reuse a stable distribution.
- **Gumbel top-k:** avoids repeated renormalization for no-replacement sampling; expected `O(m)` selection plus `O(s log s)` ordering.
- **Library answer:** if permitted, `Generator.choice(..., p=..., replace=...)` is the maintainable default. Explain the underlying method only if asked to implement it.

Product questions still matter: should weight mean probability, rank score, budget, or exposure target? A technically correct sampler can amplify popularity or starve new items.

## 2. Coupon recommendation

**Prompt:** For each user, recommend the highest-scoring eligible coupons.

Contract used here:

- `user_features`: `(n_users, d)`; `coupon_features`: `(n_coupons, d)`
- score = user/coupon dot product + coupon bias
- `eligible`: `(n_users, n_coupons)` hard business-rule mask
- return coupon column positions and aligned scores, sorted descending
- use `-1` / `-inf` padding when a user has fewer than `k` eligible coupons

Ask about the real objective (redemption, revenue, retention), already-seen coupons, budgets, diversity, cold start, tie policy, latency, and whether scores are calibrated.

In [4]:
def recommend_top_k(user_features, coupon_features, eligible, k, *, coupon_bias=None):
    """Return fixed-width top-k coupon positions and scores for every user.

    Invalid slots are padded with position ``-1`` and score ``-inf``.
    Equal-score boundary ties are unspecified unless the product supplies a tie rule.
    """
    users = np.asarray(user_features, dtype=np.float64)
    coupons = np.asarray(coupon_features, dtype=np.float64)
    eligible = np.asarray(eligible)

    if users.ndim != 2 or coupons.ndim != 2:
        raise ValueError("user_features and coupon_features must be 2D")
    if users.shape[1] != coupons.shape[1]:
        raise ValueError("user and coupon feature dimensions must match")
    expected_mask_shape = (users.shape[0], coupons.shape[0])
    if eligible.shape != expected_mask_shape:
        raise ValueError(f"eligible must have shape {expected_mask_shape}")
    if not np.all(np.isfinite(users)) or not np.all(np.isfinite(coupons)):
        raise ValueError("features must be finite")
    if isinstance(k, (bool, np.bool_)) or not isinstance(k, (int, np.integer)) or k < 0:
        raise ValueError("k must be a non-negative integer")

    n_users, n_coupons = expected_mask_shape
    if coupon_bias is None:
        bias = np.zeros(n_coupons, dtype=np.float64)
    else:
        bias = np.asarray(coupon_bias, dtype=np.float64)
        if bias.shape != (n_coupons,) or not np.all(np.isfinite(bias)):
            raise ValueError(f"coupon_bias must be finite with shape {(n_coupons,)}")

    width = min(k, n_coupons)  # Rank at most the catalog size; pad back to k below if needed.
    if width == 0:
        return (
            np.full((n_users, k), -1, dtype=np.int64),
            np.full((n_users, k), -np.inf),
        )

    scores = users @ coupons.T + bias  # Dense score matrix: (n_users, n_coupons).
    scores = np.where(
        eligible.astype(bool, copy=False), scores, -np.inf
    )  # Hard-mask ineligible coupons before ranking.

    if width == n_coupons:
        candidates = np.tile(
            np.arange(n_coupons), (n_users, 1)
        )  # Every coupon is a candidate when k covers the catalog.
    else:
        candidates = np.argpartition(-scores, kth=width - 1, axis=1)[
            :, :width
        ]  # Keep top candidates per user; order is arbitrary.
    candidate_scores = np.take_along_axis(
        scores, candidates, axis=1
    )  # Gather scores aligned to candidate positions.
    order = np.argsort(
        -candidate_scores, axis=1, kind="stable"
    )  # Sort only the width-sized candidate set.
    positions = np.take_along_axis(
        candidates, order, axis=1
    )  # Recover original coupon-column positions.
    selected_scores = np.take_along_axis(
        candidate_scores, order, axis=1
    )  # Keep scores aligned with positions.

    invalid = ~np.isfinite(
        selected_scores
    )  # Masked -inf entries represent missing recommendations.
    positions = np.where(invalid, -1, positions)
    if k > width:
        # Preserve the fixed (n_users, k) output contract when k exceeds catalog size.
        positions = np.pad(positions, ((0, 0), (0, k - width)), constant_values=-1)
        selected_scores = np.pad(selected_scores, ((0, 0), (0, k - width)), constant_values=-np.inf)
    return positions, selected_scores

In [5]:
users = np.array([[1, 0], [0, 1], [1, -1], [1, 1]], dtype=float)  # (4 users, 2 features)
coupons = np.array([[1, 0], [0, 1], [0.8, 0.8], [-1, 0]], dtype=float)  # (4 coupons, 2 features)
bias = np.array([0.0, 0.05, 0.1, -0.2])  # (4 coupons,)
eligible = np.array(
    [
        [True, True, True, False],
        [True, False, True, True],
        [False, True, False, False],
        [False, False, False, False],
    ]
)  # (4 users, 4 coupons)

coupon_positions, coupon_scores = recommend_top_k(users, coupons, eligible, k=2, coupon_bias=bias)

expected_positions = np.array([[0, 2], [2, 0], [1, -1], [-1, -1]])
assert np.array_equal(coupon_positions, expected_positions)
assert np.all(np.diff(coupon_scores[:2], axis=1) <= 0)
assert coupon_scores[2, 1] == -np.inf and np.all(coupon_scores[3] == -np.inf)
assert recommend_top_k(users, coupons, eligible, 0)[0].shape == (4, 0)
assert_raises(ValueError, recommend_top_k, users, coupons[:, :1], eligible, 2)

show("Coupon recommendation | top-k positions (-1 means no coupon)", coupon_positions)
show("Coupon recommendation | aligned scores", coupon_scores)
show(
    "Coupon recommendation | test status", "ranking, masking, padding, k=0, and shape checks passed"
)


--- Coupon recommendation | top-k positions (-1 means no coupon) ---
[[ 0  2]
 [ 2  0]
 [ 1 -1]
 [-1 -1]]

--- Coupon recommendation | aligned scores ---
[[ 1.    0.9 ]
 [ 0.9   0.  ]
 [-0.95  -inf]
 [ -inf  -inf]]

--- Coupon recommendation | test status ---
ranking, masking, padding, k=0, and shape checks passed


## 3. Connect ranking to sampling

A recommender may exploit the top score most of the time but sample among eligible coupons for exploration. Convert scores to weights only after applying hard eligibility rules. Subtracting the maximum makes exponentiation stable; temperature controls concentration. This is a product policy, not an automatic improvement.

In [6]:
user_index = 0
temperature = 0.5
all_scores = users[user_index] @ coupons.T + bias
eligible_positions = np.flatnonzero(eligible[user_index])  # Preserve mapping to catalog positions.
eligible_scores = all_scores[eligible_positions]
exploration_weights = np.exp(
    (eligible_scores - eligible_scores.max()) / temperature
)  # Stable softmax numerator; lower temperature sharpens preference.
local_draws = weighted_sample_indices(
    exploration_weights, 12, replace=True, rng=np.random.default_rng(11)
)
sampled_coupon_positions = eligible_positions[
    local_draws
]  # Convert eligible-subset draws back to catalog positions.

assert np.all(eligible[user_index, sampled_coupon_positions])
show("Exploration | eligible coupon positions", eligible_positions)
show("Exploration | stable score-derived weights", exploration_weights)
show("Exploration | seeded sampled coupon positions", sampled_coupon_positions)


--- Exploration | eligible coupon positions ---
[0 1 2]

--- Exploration | stable score-derived weights ---
[1.    0.15  0.819]

--- Exploration | seeded sampled coupon positions ---
[0 0 2 0 0 2 0 0 2 2 0 1]


### Recommendation tradeoffs at scale

The dense score matrix costs `O(n_users * n_coupons * d)` time and `O(n_users * n_coupons)` memory. `argpartition` reduces ranking from a full `O(m log m)` sort per user to expected `O(m) + O(k log k)`, but it does not remove dense scoring.

For a large catalog: apply cheap eligibility filters first, process users in batches, generate a small candidate set (rules/index/approximate nearest neighbors), then exact-score and rerank it. Cache static coupon features. Monitor not just offline accuracy but latency, redemption/revenue, coverage, diversity, exposure fairness, budget depletion, and drift. Call out the consistency tradeoff if inventory or coupon budgets change during ranking.

## 4. Optional pandas insurance (5 minutes)

Pandas was not confirmed by the recruiter, but these four operations cover a likely event-table follow-up: `groupby().agg`, `transform`, validated `merge`, and chronological `shift`. State what one row represents before manipulating it.

In [7]:
events = pd.DataFrame(
    {
        "user_id": [1, 1, 1, 2, 2, 3],
        "coupon_id": [10, 11, 10, 10, 12, 11],
        "redeemed": [1, 0, 1, 0, 1, 0],
        "revenue": [18.0, 0.0, 22.0, 0.0, 30.0, 0.0],
        "timestamp": pd.to_datetime(
            ["2026-08-01", "2026-08-03", "2026-08-08", "2026-08-02", "2026-08-09", "2026-08-04"]
        ),
    }
)
catalog = pd.DataFrame({"coupon_id": [10, 11, 12], "category": ["food", "travel", "food"]})

coupon_metrics = (
    events.groupby(
        "coupon_id", as_index=False
    )  # Collapse event rows to one summary row per coupon.
    .agg(
        impressions=("redeemed", "size"),
        redemptions=("redeemed", "sum"),
        revenue=("revenue", "sum"),
    )
    .assign(redemption_rate=lambda frame: frame["redemptions"] / frame["impressions"])
)
events = events.assign(
    user_redemption_rate=events.groupby("user_id")["redeemed"].transform("mean")
)  # transform preserves event-row alignment.
enriched = events.merge(
    catalog, on="coupon_id", how="left", validate="many_to_one"
)  # Reject catalog duplicates that would multiply rows.
enriched = enriched.sort_values(["user_id", "timestamp"]).assign(
    previous_coupon=lambda frame: frame.groupby("user_id")["coupon_id"].shift(
        1
    )  # Lag within user after chronological sorting.
)

assert len(enriched) == len(events)
assert coupon_metrics.loc[coupon_metrics["coupon_id"] == 10, "redemptions"].item() == 2
show("pandas | coupon aggregates", coupon_metrics.to_string(index=False))
show("pandas | validated join plus prior coupon", enriched.to_string(index=False))


--- pandas | coupon aggregates ---
 coupon_id  impressions  redemptions  revenue  redemption_rate
        10            3            2     40.0         0.666667
        11            2            0      0.0         0.000000
        12            1            1     30.0         1.000000

--- pandas | validated join plus prior coupon ---
 user_id  coupon_id  redeemed  revenue  timestamp  user_redemption_rate category  previous_coupon
       1         10         1     18.0 2026-08-01              0.666667     food              NaN
       1         11         0      0.0 2026-08-03              0.666667   travel             10.0
       1         10         1     22.0 2026-08-08              0.666667     food             11.0
       2         10         0      0.0 2026-08-02              0.500000     food              NaN
       2         12         1     30.0 2026-08-09              0.500000     food             10.0
       3         11         0      0.0 2026-08-04              0.000000  

## 5. Retrieval pass and interview checklist

### 20-minute emergency pass

1. **3 min:** recite the round contract and six clarifying questions.
2. **7 min:** retype the replacement sampler (`cumsum` + `searchsorted`) and three edge tests.
3. **7 min:** retype masked dot-product scoring and top-k (`argpartition` + aligned gather + sort).
4. **3 min:** explain when dense scoring fails and how candidate generation changes the design.

### During the round

- Restate the contract and label shapes before coding.
- Keep a correct baseline runnable; make one change at a time.
- Narrate the invariant you are preserving, not every keystroke.
- Run small examples and edge cases early; do not wait for the end.
- If stuck, shrink the input, write the brute-force version, inspect one intermediate value, and regain forward momentum.
- When challenged, compare alternatives by correctness, complexity, maintainability, and product/system impact.
- Finish by summarizing complexity, tests, remaining risks, and the next scale improvement.

### Two blank-page mock prompts

1. Implement weighted sampling for item IDs; add no-replacement support and explain repeated-query optimization.
2. Recommend `k` coupons per user subject to eligibility and budget constraints; test users with zero/fewer-than-`k` candidates and redesign for one million coupons.